In [6]:
import pandas as pd, numpy as np, ast

# --- tool scores + variant type (training-annotated reference) ---
onco = pd.read_csv("exonic_toolscores_oncokb_apicall.txt", sep="\t", low_memory=False)
onco = onco[onco["ANNOTATED"] == True]

tool_cols = ["VEST4_score","REVEL_score","MutPred_score","PrimateAI_score",
             "VARITY_R_score","VARITY_ER_score","ESM1b_score","EVE_score","AlphaMissense_score"]
for c in tool_cols:
    onco[c] = pd.to_numeric(onco[c], errors="coerce")

# --- LOFTEE calls for the indel arm ---
lof = pd.read_csv("sites_biallelic_exome.indels.vep112_loftee.plof_calls.tsv",
                  sep="\t", low_memory=False)
lof1 = lof[lof["mane_select"] != "."].drop_duplicates("variant_id").copy()

def vcf_to_annovar(pos, ref, alt):
    pos, ref, alt = int(pos), str(ref), str(alt)
    if len(ref) > len(alt):
        return pos + len(alt), ref[len(alt):], "-"
    elif len(alt) > len(ref):
        return pos + len(ref) - 1, "-", alt[len(ref):]
    return pos, ref, alt

conv = lof1.apply(lambda r: vcf_to_annovar(r["pos"], r["ref"], r["alt"]), axis=1)
lof1["Chr"] = lof1["chrom"]
lof1[["Start","Ref","Alt"]] = pd.DataFrame(conv.tolist(), index=lof1.index)
loftee = lof1[["Chr","Start","Ref","Alt","loftee_lof"]]

# --- population-enriched variants ---
fisher = pd.read_csv("fisher_enrichment_results.tsv.gz", sep="\t", low_memory=False)
fisher["Chr"] = fisher["locus"].str.split(":").str[0]
fisher["Start"] = fisher["locus"].str.split(":").str[1].astype(int)
parsed = fisher["alleles"].apply(ast.literal_eval)
fisher["Ref"] = parsed.str[0]
fisher["Alt"] = parsed.str[1]

print("annotated:", len(onco), "| loftee:", len(loftee),
      "| fisher unique:", fisher[["Chr","Start","Ref","Alt"]].drop_duplicates().shape[0])

annotated: 169360 | loftee: 7056 | fisher unique: 109471


In [7]:
# 1. Unique variants only (Fisher rows repeat across populations/comparisons)
fvar = fisher[["Chr","Start","Ref","Alt"]].drop_duplicates().copy()

# 2. Flag indels -- only these need notation conversion
r = fvar["Ref"].astype(str); a = fvar["Alt"].astype(str)
fvar["is_indel"] = (r.str.len() != a.str.len())
print("unique fisher variants:", len(fvar))
print("  SNVs:", int((~fvar["is_indel"]).sum()), "| indels:", int(fvar["is_indel"].sum()))

# 3. Merge against the annotated data
keep = ["Chr","Start","Ref","Alt","Gene.refGeneWithVer","Func.refGeneWithVer",
        "ExonicFunc.refGeneWithVer","ONCOGENIC"] + tool_cols
m = fvar.merge(onco[keep], on=["Chr","Start","Ref","Alt"], how="left")

# 4. Match rate, split by variant type
m["matched"] = m["ExonicFunc.refGeneWithVer"].notna()
print("\nmatched overall:", int(m["matched"].sum()), "of", len(m))
print(m.groupby("is_indel")["matched"].agg(["sum","count","mean"]).round(3))

# 5. Variant types among the matched
print("\nvariant types among matched:")
print(m.loc[m["matched"], "ExonicFunc.refGeneWithVer"].value_counts())

unique fisher variants: 109471
  SNVs: 108726 | indels: 745

matched overall: 14622 of 109471
            sum   count   mean
is_indel                      
False     14622  108726  0.134
True          0     745  0.000

variant types among matched:
ExonicFunc.refGeneWithVer
nonsynonymous SNV    13926
stopgain               554
synonymous SNV          99
startloss               42
stoploss                 1
Name: count, dtype: int64


In [8]:
# Apply the same VCF -> ANNOVAR conversion used for LOFTEE
fvar_conv = fvar.copy()
idx = fvar_conv["is_indel"]

conv = fvar_conv.loc[idx].apply(
    lambda r: vcf_to_annovar(r["Start"], r["Ref"], r["Alt"]), axis=1)
fvar_conv.loc[idx, ["Start","Ref","Alt"]] = pd.DataFrame(
    conv.tolist(), index=conv.index, columns=["Start","Ref","Alt"])

# Re-merge with converted coordinates
m2 = fvar_conv.merge(onco[keep], on=["Chr","Start","Ref","Alt"], how="left")
m2["matched"] = m2["ExonicFunc.refGeneWithVer"].notna()

print("after conversion:")
print(m2.groupby("is_indel")["matched"].agg(["sum","count","mean"]).round(3))
print("\nvariant types among matched:")
print(m2.loc[m2["matched"], "ExonicFunc.refGeneWithVer"].value_counts())

after conversion:
            sum   count   mean
is_indel                      
False     14622  108726  0.134
True        444     745  0.596

variant types among matched:
ExonicFunc.refGeneWithVer
nonsynonymous SNV          13926
stopgain                     586
frameshift deletion          279
frameshift insertion         114
synonymous SNV                99
startloss                     42
nonframeshift deletion        17
nonframeshift insertion        2
stoploss                       1
Name: count, dtype: int64


In [9]:
m3 = m2.merge(loftee, on=["Chr","Start","Ref","Alt"], how="left")

lof_types = ["frameshift deletion","frameshift insertion","stopgain","startloss","stoploss"]
sub = m3[m3["ExonicFunc.refGeneWithVer"].isin(lof_types)]

print("LOFTEE coverage by variant type:")
print(sub.groupby("ExonicFunc.refGeneWithVer")["loftee_lof"]
         .agg(n="size", with_call=lambda s: s.notna().sum(),
              hc=lambda s: (s == "HC").sum()))

LOFTEE coverage by variant type:
                             n  with_call   hc
ExonicFunc.refGeneWithVer                     
frameshift deletion        279        279  270
frameshift insertion       114        114  113
startloss                   42          0    0
stopgain                   586         31   30
stoploss                     1          0    0


In [13]:
res = m3.copy()

# --- 1. Assign a route to every variant ---
def assign_route(row):
    if pd.notna(row.get("loftee_lof")):
        return "loftee"                 # has a LOFTEE call
    et = row["ExonicFunc.refGeneWithVer"]
    if et == "nonsynonymous SNV":
        return "model"                  # missense -> ensemble
    if et == "stopgain":
        return "rule_lof"               # truncating, no LOFTEE call
    if pd.isna(et):
        return "unmatched"              # synonymous, in-frame, startloss, stoploss
    return "no_prediction"

res["route"] = res.apply(assign_route, axis=1)

# --- 2. Fill in the non-model arms ---
res["call"] = np.nan
res.loc[res["route"]=="loftee", "call"] = (
    res.loc[res["route"] == "loftee", "loftee_lof"].eq("HC").astype(int))
res.loc[res["route"] == "rule_lof", "call"] = 1          # stopgain treated as LoF

res["route"].value_counts()

route
unmatched        94393
model            13926
rule_lof           555
loftee             437
no_prediction      160
Name: count, dtype: int64